# 🎙️ VoxPoly Studio V3.0 Pro · Google Colab 一键云端版

支持 **Qwen3-TTS (1.7B / 0.6B)**、**Chatterbox Multilingual V3 (23 语种)**、**Fun-CosyVoice 3.0 (方言与情绪)** 三引擎零冲突切换，集成 **双人播客 & PDF/EPUB/URL 有声书**、**专有名词发音词典**、**广播级 LUFS 母带**、**视频 URL 直抓 + 多说话人分轨 + 样式双语 ASS 字幕 + 可选 MuseTalk 唇形同步**、**OpenAI `/v1/audio/speech` 兼容接口** 与 **AI 隐形防伪水印**。

> 顶部菜单 `代码执行程序 → 更改运行时类型` 选择 **`L4` / `A100`**（或 `T4` 自动 `fp16`），按顺序运行下方 4 个 Cell 即可！


## Step 1 · 挂载 Google Drive 并获取 VoxPoly Studio V3 代码


In [ ]:
#@title 1️⃣ 挂载 Google Drive + 检测 GPU + 拉取 V3 代码（支持 Git Clone 或上传 ZIP）
import os, shutil, subprocess, zipfile
from pathlib import Path
import torch
from google.colab import drive, files

GIT_REPO_URL = "https://github.com/zianxu78-ux/voxpoly-studio.git" #@param {type:"string"}
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/VoxPolyStudio" #@param {type:"string"}
APP_DIR = Path("/content/voxpoly_app")

drive.mount("/content/drive", force_remount=False)
drive_root = Path(DRIVE_PROJECT_DIR)
drive_data = drive_root / "data"
drive_data.mkdir(parents=True, exist_ok=True)
print(f"✅ Google Drive 已挂载，模型与作品持久化目录: {drive_data}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    cc_major, cc_minor = torch.cuda.get_device_capability(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    precision = "bf16 + TF32" if cc_major >= 8 else "fp16 (T4/V100 自动兼容)"
    print(f"🚀 GPU: {gpu_name} ({total_gb:.1f} GB VRAM, sm_{cc_major}{cc_minor}) → 精度: {precision}")
else:
    print("⚠️ 未检测到 GPU，请先切换为 GPU 运行时。")

if APP_DIR.exists():
    shutil.rmtree(APP_DIR)

if GIT_REPO_URL.strip():
    subprocess.run(["git", "clone", "--depth", "1", GIT_REPO_URL.strip(), str(APP_DIR)], check=True)
    print(f"✅ 已从 Git 仓库克隆至 {APP_DIR}")
else:
    candidates = [drive_root / "voxpoly_colab_bundle.zip", Path("/content/voxpoly_colab_bundle.zip")]
    bundle_zip = next((p for p in candidates if p.exists()), None)
    if bundle_zip is None:
        print("📤 请填写上方 GIT_REPO_URL，或直接在此上传 voxpoly_colab_bundle.zip：")
        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith(".zip"):
                dst = drive_root / "voxpoly_colab_bundle.zip"
                shutil.copy2(Path("/content") / fname, dst)
                bundle_zip = dst
                break
    assert bundle_zip and bundle_zip.exists(), "❌ 未提供仓库地址或代码压缩包。"
    APP_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle_zip, "r") as zf:
        zf.extractall(APP_DIR)
    print(f"✅ 代码已解压至 {APP_DIR}")


## Step 2 · 安装 V3 依赖（三引擎 + 视频抓流/分说话人/Demucs + 可选 MuseTalk 唇形同步）


In [ ]:
#@title 2️⃣ 安装依赖（复用 Colab CUDA PyTorch，各引擎及 Lip-Sync 隔离零冲突）
import os, sys, subprocess
from pathlib import Path

WHISPER_MODEL = "large-v3-turbo" #@param ["large-v3-turbo", "small", "medium"]
ENABLE_DEMUCS_VIDEO_BGM = True #@param {type:"boolean"}
ENABLE_CHATTERBOX_23LANG = True #@param {type:"boolean"}
ENABLE_COSYVOICE3_DIALECT = True #@param {type:"boolean"}
ENABLE_LIPSYNC_MUSETALK = False #@param {type:"boolean"}

def run(cmd: str, check: bool = True):
    print(f"▶ {cmd}")
    res = subprocess.run(cmd, shell=True)
    if check and res.returncode != 0:
        raise RuntimeError(f"命令失败 (exit={res.returncode}): {cmd}")
    return res.returncode

def ensure_venv(venv_dir: Path):
    if (venv_dir / "bin" / "python").exists():
        return
    if run(f"{sys.executable} -m venv --system-site-packages {venv_dir}", check=False) != 0:
        run(f"{sys.executable} -m pip install -q virtualenv")
        run(f"virtualenv --system-site-packages {venv_dir}")

run("apt-get -qq update && apt-get -qq install -y ffmpeg fonts-noto-cjk sox libsox-dev")

core_pkgs = (
    "qwen-tts huggingface_hub modelscope "
    "fastapi 'uvicorn[standard]' python-multipart "
    "soundfile librosa scipy scikit-learn faster-whisper deep-translator httpx "
    "pypdf beautifulsoup4 yt-dlp"
)
if ENABLE_DEMUCS_VIDEO_BGM:
    core_pkgs += " demucs"
run(f"{sys.executable} -m pip install -q --upgrade {core_pkgs}")

if ENABLE_CHATTERBOX_23LANG:
    cb_venv = Path("/content/chatterbox_env")
    ensure_venv(cb_venv)
    run(
        f"{cb_venv}/bin/pip install -q --ignore-installed "
        "transformers==5.2.0 diffusers==0.29.0 conformer==0.3.2 "
        "s3tokenizer spacy-pkuseg pykakasi==2.3.0 pyloudnorm omegaconf safetensors soundfile"
    )
    run(f"{cb_venv}/bin/pip install -q 'git+https://github.com/resemble-ai/Perth.git@master'")
    run(f"{cb_venv}/bin/pip install -q --no-deps chatterbox-tts")

if ENABLE_COSYVOICE3_DIALECT:
    cv_repo = Path("/content/CosyVoice")
    if not cv_repo.exists():
        run(f"git clone --depth 1 --recursive https://github.com/FunAudioLLM/CosyVoice.git {cv_repo}")
    cv_venv = Path("/content/cosyvoice_env")
    ensure_venv(cv_venv)
    run(
        f"{cv_venv}/bin/pip install -q "
        "conformer diffusers gdown hydra-core HyperPyYAML inflect librosa lightning "
        "modelscope huggingface_hub omegaconf onnxruntime-gpu openai-whisper pyworld "
        "rich soundfile wetext x-transformers"
    )

if ENABLE_LIPSYNC_MUSETALK:
    ls_repo = Path("/content/MuseTalk")
    if not ls_repo.exists():
        run(f"git clone --depth 1 https://github.com/TMElyralab/MuseTalk.git {ls_repo}")
    ls_venv = Path("/content/lipsync_env")
    ensure_venv(ls_venv)
    run(f"{ls_venv}/bin/pip install -q diffusers accelerate einops opencv-python-headless imageio imageio-ffmpeg omegaconf pyyaml", check=False)
    print("ℹ️ 提示：首次使用 MuseTalk 唇形同步时请按 MuseTalk README 下载权重到 /content/MuseTalk/models/")

print("✅ 全部 V3 所选依赖安装完成！")


## Step 3 · 启动 VoxPoly Studio V3 后端服务（含 OpenAI `/v1/audio/speech` API）


In [ ]:
#@title 3️⃣ 启动服务（端口 8765，支持访问口令、OpenAI 兼容接口与 Lip-Sync 路由）
import os, sys, time, json, subprocess, urllib.request
from pathlib import Path

DRIVE_DATA_DIR = "/content/drive/MyDrive/VoxPolyStudio/data" #@param {type:"string"}
QWEN_VARIANT = "1.7b" #@param ["1.7b", "0.6b"]
WHISPER_MODEL = "large-v3-turbo" #@param ["large-v3-turbo", "small", "medium"]
QWEN_BATCH_SIZE = 3 #@param {type:"slider", min:1, max:4, step:1}
ACCESS_TOKEN = "" #@param {type:"string"}
PRELOAD_QWEN_ON_START = True #@param {type:"boolean"}
PORT = 8765

subprocess.run("pkill -f 'uvicorn backend.main:app' || true", shell=True)
time.sleep(1)

env = os.environ.copy()
env["VOXPOLY_DATA_DIR"] = DRIVE_DATA_DIR
env["VOXPOLY_DOWNLOAD_SOURCE"] = "hf"
env["VOXPOLY_QWEN_VARIANT"] = QWEN_VARIANT
env["VOXPOLY_WHISPER_MODEL"] = WHISPER_MODEL
env["VOXPOLY_WHISPER_DEVICE"] = "auto"
env["VOXPOLY_QWEN_BATCH_SIZE"] = str(QWEN_BATCH_SIZE)
env["VOXPOLY_ACCESS_TOKEN"] = ACCESS_TOKEN.strip()
env["VOXPOLY_CHATTERBOX_PYTHON"] = "/content/chatterbox_env/bin/python"
env["VOXPOLY_COSYVOICE_PYTHON"] = "/content/cosyvoice_env/bin/python"
env["VOXPOLY_COSYVOICE_REPO"] = "/content/CosyVoice"
env["VOXPOLY_LIPSYNC_PYTHON"] = "/content/lipsync_env/bin/python"
env["VOXPOLY_LIPSYNC_REPO"] = "/content/MuseTalk"
env["VOXPOLY_LIPSYNC_BACKEND"] = "musetalk"
env["HF_HOME"] = str(Path(DRIVE_DATA_DIR) / "models" / "hf_cache")
env["PYTHONUNBUFFERED"] = "1"

log_path = Path("/content/voxpoly_server.log")
log_file = open(log_path, "w", encoding="utf-8")
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "backend.main:app", "--host", "127.0.0.1", "--port", str(PORT)],
    cwd="/content/voxpoly_app",
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

headers = {"X-Access-Token": ACCESS_TOKEN.strip()} if ACCESS_TOKEN.strip() else {}
status = None
for _ in range(30):
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{PORT}/api/system/status", headers=headers)
        with urllib.request.urlopen(req, timeout=2) as r:
            status = json.loads(r.read().decode("utf-8"))
            break
    except Exception:
        time.sleep(1)

assert status, f"启动失败:\n{log_path.read_text(encoding='utf-8', errors='replace')[-2000:]}"
feats = status.get("features", {})
print(f"✅ VoxPoly V3 就绪 (PID={proc.pid}) | GPU: {status['engine']['gpu']['name']} ({status['engine']['gpu']['recommended_precision']})")
print(f"   yt-dlp={feats.get('yt_dlp_available')} | Demucs={feats.get('demucs_available')} | Lip-Sync={feats.get('lipsync_available')} | OpenAI API=/v1/audio/speech")

if PRELOAD_QWEN_ON_START:
    req = urllib.request.Request(
        f"http://127.0.0.1:{PORT}/api/system/preload",
        data=json.dumps({"engine_id": "qwen3"}).encode("utf-8"),
        headers={**headers, "Content-Type": "application/json"},
        method="POST",
    )
    urllib.request.urlopen(req, timeout=10)
    print("⏳ 正在后台预热 Qwen3-TTS 模型...")


## Step 4 · 打开网页工作台（Colab 官方免密 HTTPS + 可选 Cloudflare 公网）


In [ ]:
#@title 4️⃣ 打开 VoxPoly Studio V3 网页工作台
import re, time, subprocess
from pathlib import Path
from google.colab import output

PORT = 8765
ENABLE_PUBLIC_CLOUDFLARE_URL = True #@param {type:"boolean"}

output.serve_kernel_port_as_window(PORT, anchor_text="🎙️ 打开 VoxPoly Studio V3.0 Pro（Colab 官方 HTTPS）")

if ENABLE_PUBLIC_CLOUDFLARE_URL:
    cf_bin = Path("/usr/local/bin/cloudflared")
    if not cf_bin.exists():
        subprocess.run(
            "wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared",
            shell=True,
            check=True,
        )
    subprocess.run("pkill -f 'cloudflared tunnel' || true", shell=True)
    cf_log = Path("/content/cloudflared.log")
    cf_f = open(cf_log, "w", encoding="utf-8")
    subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"], stdout=cf_f, stderr=subprocess.STDOUT)
    for _ in range(25):
        time.sleep(1)
        m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", cf_log.read_text(encoding="utf-8", errors="replace"))
        if m:
            pub = m.group(0)
            print(f"\n🌐 Cloudflare 临时公网 HTTPS 链接:\n👉 Web 工作台: {pub}\n👉 OpenAI TTS Base URL: {pub}/v1")
            break
